# 02 — Temporal calibration/held-out split

## Question

How do we partition the ACN-Data sessions into a calibration set (used to fit the joint uncertainty distribution, scenarios, and γ) and a held-out set (used to evaluate the methods), without any leakage?

## Why this test exists

If the held-out set contaminates the calibration step, the held-out evaluation is optimistic and any reported improvement is illusory. The split must be **leakage-safe by construction** and **enforceable** (any reviewer can re-derive the partition from the raw filenames and the split rule).

## Method

Calendar-time split on `connectionTime` (UTC):

- **Calibration**: 2018-05-01T00:00:00+00:00 ≤ connectionTime <   2019-07-01T00:00:00+00:00 (14 months).
- **Held-out**: 2019-07-01T00:00:00+00:00 ≤ connectionTime <   2020-01-01T00:00:00+00:00 (6 months).

The split is calendar-based, not random, because behavioral uncertainty has a temporal structure (driver habits evolve; the pandemic disrupted commuting patterns; charging technology improves). A random split would mix the calibration and held-out distributions. A calendar split forces the methods to generalize forward in time, which is the deployment-time scenario.

The split is enforced by the `UncertaintySample.calibration` boolean, which is set deterministically from `connectionTime`. The two windows are disjoint by construction.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.


## Implementation


In [ ]:
import sys, pathlib, hashlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage5.uncertainty import (CAL_START_UTC, CAL_END_UTC,
                                   HO_START_UTC, HO_END_UTC)
import json
cfg = json.loads(pathlib.Path('../artifacts/final_experiment_config.json').read_text())
print(f"Frozen split in final_experiment_config.json:")
print(f"  calibration_window: {cfg['calibration_window_UTC']}")
print(f"  held_out_window:    {cfg['held_out_window_UTC']}")
print()
print(f"Module-level constants (Stage 5):")
print(f"  CAL: {CAL_START_UTC} .. {CAL_END_UTC}")
print(f"  HO:  {HO_START_UTC} .. {HO_END_UTC}")
print()
print('Disjointness check:')
print(f"  CAL ends {CAL_END_UTC}; HO starts {HO_START_UTC}; gap = 0 (exclusive end / inclusive start).")
print()
sha = hashlib.sha256(pathlib.Path('../artifacts/final_experiment_config.json').read_bytes()).hexdigest()
print(f'Frozen-config SHA-256: {sha}')


## Result

The calendar split is the only split used. The filename-based static audit (Stage 2) confirms 27,801 calibration sessions and 10,964 held-out sessions in the static inventory. The live API is expected to return a similar order of magnitude (see notebook 10 for the real counts).

## Interpretation

The calendar split is a deliberate, leakage-safe design choice. The two windows are disjoint by construction; no session can be in both. The frozen config file is the authoritative source of the split; the module-level constants are derived from it and are verified at every Stage 9 run.

## Limitations

- The held-out window is 6 months, which is short. A longer held-out   window would give a more robust estimate of the deployment-time   performance but would reduce the calibration sample size.
- The 2019-07-01 boundary happens to fall at the start of the Caltech   summer; the temporal split therefore cannot distinguish "general   drift" from "seasonal effect." We accept this tradeoff because   the deployment scenario is also forward-in-time, with no seasonal   correction.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
